<div align="center">

# <span style="color:#2E86C1;"> Advanced Tutorial 1: The Art of Optimization: From Mountains to Money 🏔️💰</span>

**Author:** [<span style="color:#8E44AD;">Dr. D Bhanu Prakash</span>](https://dbhanuprakash233.github.io)

<img src="https://github.com/dbhanuprakash233/SSSIHL_DBP/blob/main/assets/SssihlLogo.png?raw=true" alt="University Logo" width="80"/>

**<span style="color:#16A085;">Sri Sathya Sai Institute of Higher Learning</span>**  
<span style="color:#5D6D7E;">Prasanthi Nilayam - 515 134, Andhra Pradesh, India.</span>

**Course:** <span style="color:#D35400;">Optimization Techniques for Machine Learning</span>  
**Course Code:** <span style="color:#1ABC9C;">UMAM-502</span>

</div>

**Imagine this:** you are hiking down a foggy mountain, blindfolded, trying to reach the lowest
valley. You can only feel the slope under your feet. Every step you take is a *decision* —
go steeper, go gentler, go left, go right. That, in a nutshell, is **optimization**.

Every time Netflix picks a movie for you, a rocket lands itself, a bank builds a portfolio,
or an airline prices a ticket — somewhere, quietly, an optimization algorithm is doing exactly
what you're about to build **from scratch** in this notebook.

## What you will build today
| Part | Topic | Real-life twist |
|---|---|---|
| 1 | Unconstrained optimization (Gradient Descent, Newton, BFGS) | Fitting a line to real house-price data |
| 2 | Set-constrained optimization (Projected GD, Penalty, Lagrange) | Building a minimum-risk investment portfolio |
| 3 | Capstone challenge | Design the cheapest possible soda can 🥤 |

## Rules of the game
1. Cells marked **`# TODO`** are for *you* to complete.
2. Cells marked **`### ✅ ANSWER`** contain a reference solution — **try the TODO first**,
   only peek after you've genuinely attempted it.
3. Every section ends with a **🤔 Think About It** question. There's no code for these —
   just your brain.

Let's descend the mountain.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy import optimize as spo

plt.rcParams['figure.figsize'] = (7, 5)
np.random.seed(42)


## Part 0 — Seeing the Landscape

Optimization is geometry. Before writing a single algorithm, let's *look* at what we're
trying to minimize: the **Himmelblau function**, a classic test function with four
equally-good minima.

$$f(x,y) = (x^2+y-11)^2 + (x+y^2-7)^2$$


In [ ]:
def himmelblau(p):
    x, y = p
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

def himmelblau_grad(p):
    x, y = p
    dfdx = 2*(x**2 + y - 11)*2*x + 2*(x + y**2 - 7)
    dfdy = 2*(x**2 + y - 11) + 2*(x + y**2 - 7)*2*y
    return np.array([dfdx, dfdy])

X, Y = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
Z = (X**2 + Y - 11)**2 + (X + Y**2 - 7)**2

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.85)
ax1.set_title("Himmelblau's function — a landscape with 4 valleys")

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(X, Y, Z, levels=30, cmap='viridis')
ax2.set_title("Contour view (top-down map)")
plt.tight_layout()
plt.show()


### 🤔 Think About It
Look at the contour plot. There are **four** dark blue basins (minima). If you drop a ball
from a random point on this surface and let it roll downhill, will it always reach the
*same* minimum? What does this tell you about gradient-based methods and **local vs.
global** optima?


---
## Part 1 — Unconstrained Optimization

**Problem statement:** find $x^* = \arg\min_{x \in \mathbb{R}^n} f(x)$ with *no* restriction on $x$.

**Necessary condition (first-order):** $\nabla f(x^*) = 0$.
**Sufficient condition (second-order):** Hessian $\nabla^2 f(x^*)$ is positive definite.

### 🤔 Think About It (before you code)
Is $\nabla f(x^*) = 0$ *enough* to guarantee $x^*$ is a minimum? Sketch (mentally) the
function $f(x,y) = x^2 - y^2$ at the origin. What kind of point is that? (This is called a
**saddle point** — keep it in mind, it will haunt every gradient method you build.)


### 1.1 Gradient Descent

The simplest idea in all of optimization: **walk downhill**.

$$x_{k+1} = x_k - \alpha \, \nabla f(x_k)$$

where $\alpha$ is the **step size / learning rate**.

**Your task:** complete `gradient_descent` below. It should stop when either the gradient
norm is below `tol` or `max_iter` is reached, and it should return the full path taken
(for plotting).


In [ ]:
def gradient_descent(f, grad_f, x0, alpha=0.001, tol=1e-6, max_iter=5000):
    """
    Minimize f starting from x0 using vanilla gradient descent.
    Returns: x_star, path (list of x's visited), history of f-values
    """
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    fvals = [f(x)]

    for i in range(max_iter):
        grad = None          # TODO: compute the gradient at x
        if np.linalg.norm(grad) < tol:
            break
        x = None              # TODO: take the gradient step
        path.append(x.copy())
        fvals.append(f(x))

    return x, np.array(path), fvals


### ✅ ANSWER

In [ ]:
def gradient_descent(f, grad_f, x0, alpha=0.001, tol=1e-6, max_iter=5000):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    fvals = [f(x)]

    for i in range(max_iter):
        grad = grad_f(x)
        if np.linalg.norm(grad) < tol:
            break
        x = x - alpha * grad
        path.append(x.copy())
        fvals.append(f(x))

    return x, np.array(path), fvals

x_star, path, fvals = gradient_descent(himmelblau, himmelblau_grad, x0=[0.0, 0.0], alpha=0.005)
print("GD found minimum at:", x_star, " f(x*) =", himmelblau(x_star), " in", len(path)-1, "steps")

plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.plot(path[:,0], path[:,1], 'r.-', markersize=3, label='GD path')
plt.scatter(*path[0], c='black', label='start')
plt.legend(); plt.title('Gradient Descent path on Himmelblau'); plt.show()


### 🤔 Think About It
Re-run gradient descent from `x0=[0.0, 0.0]` starting with `alpha=0.005`, then try
`alpha=0.02`. What happens? Then try `alpha=0.2`. At what point does the algorithm
**diverge**, and why do you think a bigger step isn't just "faster convergence"?


### 1.2 Newton's Method

Gradient descent only uses *slope*. Newton's method also uses *curvature* (the Hessian),
approximating $f$ locally by a quadratic and jumping straight to its minimum:

$$x_{k+1} = x_k - [\nabla^2 f(x_k)]^{-1} \nabla f(x_k)$$

**Your task:** complete `newton_method`. You are given the Hessian function.


In [ ]:
def himmelblau_hess(p):
    x, y = p
    d2fdx2 = 12*x**2 + 4*y - 42
    d2fdy2 = 4*x + 12*y**2 - 26
    d2fdxdy = 4*x + 4*y
    return np.array([[d2fdx2, d2fdxdy], [d2fdxdy, d2fdy2]])

def newton_method(f, grad_f, hess_f, x0, tol=1e-8, max_iter=100):
    x = np.array(x0, dtype=float)
    path = [x.copy()]

    for i in range(max_iter):
        grad = grad_f(x)
        if np.linalg.norm(grad) < tol:
            break
        H = None            # TODO: compute the Hessian at x
        step = None          # TODO: solve H @ step = grad  (use np.linalg.solve)
        x = None              # TODO: update x using the Newton step
        path.append(x.copy())

    return x, np.array(path)


### ✅ ANSWER

In [ ]:
def newton_method(f, grad_f, hess_f, x0, tol=1e-8, max_iter=100):
    x = np.array(x0, dtype=float)
    path = [x.copy()]

    for i in range(max_iter):
        grad = grad_f(x)
        if np.linalg.norm(grad) < tol:
            break
        H = hess_f(x)
        step = np.linalg.solve(H, grad)
        x = x - step
        path.append(x.copy())

    return x, np.array(path)

xN, pathN = newton_method(himmelblau, himmelblau_grad, himmelblau_hess, x0=[0.0, 3.5])
print("Newton found minimum at:", xN, "in", len(pathN)-1, "steps (compare to GD's ~hundreds!)")

plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.plot(pathN[:,0], pathN[:,1], 'g.-', markersize=6, label='Newton path')
plt.legend(); plt.title("Newton's Method — notice how few steps it needs"); plt.show()


### 🤔 Think About It
Newton's method converges in a handful of steps versus hundreds for gradient descent.
So why doesn't everyone just use Newton's method all the time? (Hint: what do you need to
compute and invert at *every single step*, and what happens when $x$ has 10,000
dimensions, like in a neural network?)


### 1.3 Quasi-Newton (BFGS) — the practical compromise

BFGS approximates the Hessian using only gradient information, avoiding the expensive
Hessian computation/inversion while still converging fast. Rather than hand-rolling it,
professionals use `scipy.optimize.minimize`.

**Your task:** call `scipy.optimize.minimize` with `method='BFGS'` to minimize `himmelblau`.


In [ ]:
# TODO: use spo.minimize with method='BFGS', starting at x0=[0.0, 0.0], jac=himmelblau_grad
result = None
print(result)


### ✅ ANSWER

In [ ]:
result = spo.minimize(himmelblau, x0=[0.0, 0.0], jac=himmelblau_grad, method='BFGS')
print(result.x, result.fun, result.nit, "iterations")


---
## 🏠 Real-Life Application 1: Linear Regression *is* Unconstrained Optimization

Every time you fit a trend line, you're solving an unconstrained optimization problem:
minimize the **Mean Squared Error** between predictions and reality.

$$\min_{w,b} \; L(w,b) = \frac{1}{n}\sum_{i=1}^n (w x_i + b - y_i)^2$$

We'll use a small synthetic "house size vs. price" dataset.


In [ ]:
# Synthetic housing data: size (1000 sqft) vs price ($100k)
size = np.array([0.8, 1.0, 1.2, 1.5, 1.8, 2.0, 2.3, 2.6, 3.0, 3.4])
price = 1.5*size + 0.8 + np.random.normal(0, 0.15, size.shape)

plt.scatter(size, price); plt.xlabel('Size (1000 sqft)'); plt.ylabel('Price ($100k)')
plt.title('House price vs size'); plt.show()


**Your task:** complete `mse_loss` and `mse_grad` (the gradient of MSE with respect to
`w` and `b`), then use YOUR `gradient_descent` function from Part 1.1 to fit the line.


In [ ]:
def mse_loss(params):
    w, b = params
    preds = None        # TODO: w*size + b
    return np.mean((preds - price)**2)

def mse_grad(params):
    w, b = params
    preds = None         # TODO
    error = None          # TODO: preds - price
    dw = None              # TODO: derivative of MSE w.r.t. w  -> (2/n) * sum(error * size)
    db = None              # TODO: derivative of MSE w.r.t. b  -> (2/n) * sum(error)
    return np.array([dw, db])

w_star, path_wb, losses = gradient_descent(mse_loss, mse_grad, x0=[0.0, 0.0], alpha=0.05, max_iter=2000)
print("Best fit: price =", w_star[0], "* size +", w_star[1])


### ✅ ANSWER

In [ ]:
def mse_loss(params):
    w, b = params
    preds = w*size + b
    return np.mean((preds - price)**2)

def mse_grad(params):
    w, b = params
    n = len(size)
    preds = w*size + b
    error = preds - price
    dw = (2/n) * np.sum(error * size)
    db = (2/n) * np.sum(error)
    return np.array([dw, db])

w_star, path_wb, losses = gradient_descent(mse_loss, mse_grad, x0=[0.0, 0.0], alpha=0.05, max_iter=2000)
print("Best fit: price =", round(w_star[0],3), "* size +", round(w_star[1],3))

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].scatter(size, price, label='data')
xs = np.linspace(0.5, 3.5, 50)
axes[0].plot(xs, w_star[0]*xs + w_star[1], 'r-', label='GD fit')
axes[0].legend(); axes[0].set_title('Fitted line')
axes[1].plot(losses); axes[1].set_title('Loss curve'); axes[1].set_xlabel('iteration'); axes[1].set_ylabel('MSE')
plt.tight_layout(); plt.show()


### 🤔 Think About It
This is *exactly* how the first layer of every neural network learns. Now imagine `price`
also depended on `bedrooms`, `location`, `age` — a 50-dimensional `w`. Would you rather
compute a 50×50 Hessian (Newton) every step, or a 50-dimensional gradient (GD/BFGS)? Why
do you think **deep learning almost never uses Newton's method**?


---
## Part 2 — Set-Constrained Optimization

**Problem statement:** find $x^* = \arg\min_{x \in C} f(x)$, where $C \subseteq \mathbb{R}^n$
is a **feasible set** (box, ball, hyperplane, simplex...).

### 🤔 Think About It (before you code)
In Part 1, the optimum always satisfied $\nabla f(x^*)=0$. If the true unconstrained
minimum lies *outside* the feasible set $C$, can $\nabla f(x^*)=0$ still hold at the
constrained optimum $x^*$? Where do you expect $x^*$ to sit — inside $C$, or on its
**boundary**?


### 2.1 Projected Gradient Descent

Idea: take a normal gradient step, then **snap back** onto the feasible set $C$ using a
projection operator $\text{Proj}_C(x) = \arg\min_{c \in C}\|x - c\|$.

$$x_{k+1} = \text{Proj}_C\big(x_k - \alpha \nabla f(x_k)\big)$$

We'll constrain Himmelblau's function to the **box** $-2 \le x,y \le 2$.

**Your task:** complete the box-projection function and the projected gradient descent loop.


In [ ]:
def project_box(x, lo, hi):
    return None   # TODO: clip x element-wise between lo and hi (use np.clip)

def projected_gradient_descent(f, grad_f, project, x0, alpha=0.005, tol=1e-6, max_iter=5000):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for i in range(max_iter):
        grad = grad_f(x)
        x_new = None        # TODO: gradient step, then project onto the feasible set
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            path.append(x.copy())
            break
        x = x_new
        path.append(x.copy())
    return x, np.array(path)


### ✅ ANSWER

In [ ]:
def project_box(x, lo, hi):
    return np.clip(x, lo, hi)

def projected_gradient_descent(f, grad_f, project, x0, alpha=0.005, tol=1e-6, max_iter=5000):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for i in range(max_iter):
        grad = grad_f(x)
        x_new = project(x - alpha * grad)
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            path.append(x.copy())
            break
        x = x_new
        path.append(x.copy())
    return x, np.array(path)

box_project = lambda z: project_box(z, -2, 2)
xB, pathB = projected_gradient_descent(himmelblau, himmelblau_grad, box_project, x0=[0.0, 0.0], alpha=0.01)
print("Constrained minimum (box [-2,2]^2):", xB, "f(x) =", himmelblau(xB))

plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.plot(pathB[:,0], pathB[:,1], 'm.-', label='Projected GD path')
plt.gca().add_patch(plt.Rectangle((-2,-2), 4, 4, fill=False, edgecolor='black', lw=2, label='feasible box'))
plt.legend(); plt.title('Projected Gradient Descent — constrained to a box'); plt.show()


### 🤔 Think About It
Change the box to `lo=-1, hi=1` (a smaller feasible region that excludes ALL four of
Himmelblau's true minima). Where does the algorithm end up? Confirm: does it sit strictly
*inside* the box, or does it get stuck on an *edge* or *corner*? Relate this to KKT
complementary slackness (constraints that "bind" have nonzero multipliers).


### 2.2 Penalty Method

Alternative idea: don't project — instead, **punish** the objective for leaving $C$, turning
a constrained problem into an unconstrained one.

$$f_{\text{penalized}}(x) = f(x) + \mu \cdot \sum_i \max(0, |x_i| - 2)^2$$

As $\mu \to \infty$, the penalized minimizer converges to the true constrained minimizer.

**Your task:** complete the penalty term for the same box $[-2,2]^2$, then minimize with
your own `gradient_descent`.


In [ ]:
def penalty_term(x, lo=-2, hi=2):
    violation_low = None    # TODO: how much x is below lo, elementwise (>=0)
    violation_high = None    # TODO: how much x is above hi, elementwise (>=0)
    return np.sum(violation_low**2 + violation_high**2)

def penalized_f(x, mu):
    return himmelblau(x) + mu * penalty_term(x)

def penalized_grad(x, mu, eps=1e-6):
    # numerical gradient of the penalized objective (finite differences)
    grad = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        xp = x.copy(); xp[i] += eps
        xm = x.copy(); xm[i] -= eps
        grad[i] = (penalized_f(xp, mu) - penalized_f(xm, mu)) / (2*eps)
    return grad


### ✅ ANSWER

In [ ]:
def penalty_term(x, lo=-2, hi=2):
    violation_low = np.maximum(0, lo - x)
    violation_high = np.maximum(0, x - hi)
    return np.sum(violation_low**2 + violation_high**2)

def penalized_f(x, mu):
    return himmelblau(x) + mu * penalty_term(x)

def penalized_grad(x, mu, eps=1e-6):
    grad = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        xp = x.copy(); xp[i] += eps
        xm = x.copy(); xm[i] -= eps
        grad[i] = (penalized_f(xp, mu) - penalized_f(xm, mu)) / (2*eps)
    return grad

for mu in [1, 10, 100, 1000]:
    xP, _, _ = gradient_descent(lambda x: penalized_f(x, mu), lambda x: penalized_grad(x, mu),
                                  x0=[0.0, 0.0], alpha=0.005, max_iter=3000)
    print(f"mu={mu:5d} -> x* = {xP}, f(x*) = {himmelblau(xP):.4f}")


### 🤔 Think About It
As `mu` grows, does the penalty-method answer converge to the same point as your
**Projected Gradient Descent** answer? What trade-off do you see for very large `mu`
(look at how the algorithm behaves, not just the endpoint — is the loss landscape
becoming easy or hard to descend)?


### 2.3 Lagrange Multipliers (equality constraint)

For a constraint set defined by an **equality**, e.g. $g(x) = x_1 + x_2 - 3 = 0$, the
optimum satisfies the **Lagrangian stationarity condition**:

$$\nabla f(x^*) = \lambda \nabla g(x^*)$$

**Your task:** use `scipy.optimize.minimize` with an equality constraint dictionary to
minimize Himmelblau subject to $x_1 + x_2 = 3$.


In [ ]:
constraint = {'type': 'eq', 'fun': None}   # TODO: define g(x) = x[0] + x[1] - 3
result_eq = None   # TODO: spo.minimize(himmelblau, x0=[1,1], constraints=[constraint])
print(result_eq)


### ✅ ANSWER

In [ ]:
constraint = {'type': 'eq', 'fun': lambda x: x[0] + x[1] - 3}
result_eq = spo.minimize(himmelblau, x0=[1.0, 1.0], constraints=[constraint])
print("Constrained (x1+x2=3) minimum:", result_eq.x, " f =", result_eq.fun)


---
## 💼 Real-Life Application 2: Minimum-Risk Portfolio (Budget-Constrained)

You have `k` assets with a covariance matrix $\Sigma$ (risk). You want to choose portfolio
weights $w$ that **minimize risk** $w^T \Sigma w$ subject to:

- $\sum_i w_i = 1$ (fully invested — this is your budget constraint)
- $w_i \ge 0$ (no short-selling)

This feasible set is called the **simplex**. We'll solve it with Projected Gradient Descent,
where projection onto the simplex is the interesting bit.


In [ ]:
# 4 synthetic assets with a covariance matrix (risk relationships)
np.random.seed(1)
A = np.random.randn(4, 4) * 0.05
Sigma = A @ A.T + np.eye(4) * 0.01   # guarantees positive semi-definite
print("Covariance matrix:\n", np.round(Sigma, 4))

def portfolio_risk(w):
    return w @ Sigma @ w

def portfolio_risk_grad(w):
    return 2 * Sigma @ w


**Your task:** complete `project_simplex`, which projects any vector `w` onto the
probability simplex $\{w : \sum w_i = 1, w_i \ge 0\}$. (This is a classic algorithm —
sort, find a threshold `theta`, then shift-and-clip.)


In [ ]:
def project_simplex(v):
    n = len(v)
    u = np.sort(v)[::-1]
    css = np.cumsum(u) - 1
    ind = np.arange(1, n+1)
    cond = None          # TODO: boolean array where u - css/ind > 0
    rho = None            # TODO: the largest index i (1-based) satisfying cond
    theta = css[rho-1] / rho
    w = None               # TODO: np.maximum(v - theta, 0)
    return w


### ✅ ANSWER

In [ ]:
def project_simplex(v):
    n = len(v)
    u = np.sort(v)[::-1]
    css = np.cumsum(u) - 1
    ind = np.arange(1, n+1)
    cond = u - css/ind > 0
    rho = ind[cond][-1]
    theta = css[rho-1] / rho
    w = np.maximum(v - theta, 0)
    return w

w_star, pathW = projected_gradient_descent(portfolio_risk, portfolio_risk_grad, project_simplex,
                                             x0=[0.25, 0.25, 0.25, 0.25], alpha=1.0, max_iter=500)
print("Optimal weights:", np.round(w_star, 4), " sum =", round(w_star.sum(), 4))
print("Minimum risk (variance):", portfolio_risk(w_star))

plt.pie(w_star, labels=[f"Asset {i+1}" for i in range(4)], autopct='%1.1f%%')
plt.title("Minimum-Risk Portfolio Allocation"); plt.show()


### 🤔 Think About It
Look at the resulting weights — is any asset allocated **exactly 0%**? That asset sits on
the *boundary* of the simplex (the non-negativity constraint is "active"). Relate this back
to your box-constraint observation in 2.1. In finance, what does a weight of exactly 0% mean
about that asset's contribution to lowering portfolio risk?


---
## 🥤 Capstone Challenge (no answer key — this one's on you!)

**The problem:** A beverage company wants to design a cylindrical can that holds a fixed
volume $V = 330\,\text{cm}^3$ while using the **minimum amount of aluminum** (surface area).

$$\min_{r,h} \; S(r,h) = 2\pi r^2 + 2\pi r h \quad \text{s.t.} \quad \pi r^2 h = V,\; r,h > 0$$

This is a **constrained optimization problem with a nonlinear equality constraint** —
exactly the shape of problem Lagrange multipliers were invented for.

**Your tasks:**
1. Write `surface_area(r, h)` and the constraint `volume_constraint(r, h, V)`.
2. Solve it with `scipy.optimize.minimize` using an equality constraint (like Section 2.3).
3. Solve it *again* using the classical calculus approach: substitute
   $h = V/(\pi r^2)$ into $S$, reducing it to a 1-D **unconstrained** problem, and use
   your `gradient_descent` or `newton_method` from Part 1.
4. **Confirm both approaches agree.**
5. Bonus: real cans have $r \approx 3.25\,\text{cm}$. How far is the "mathematically
   optimal" can from a real one — and what non-mathematical constraints (manufacturing,
   ergonomics, shelf design) might explain the gap?


In [ ]:
# Your workspace — go!
V = 330  # cm^3

def surface_area(params):
    r, h = params
    pass  # TODO

def volume_constraint(params, V=V):
    r, h = params
    pass  # TODO

# TODO: Approach A - scipy.optimize.minimize with an equality constraint

# TODO: Approach B - reduce to 1D via h = V/(pi r^2), then apply gradient_descent or newton_method

# TODO: Compare r*, h*, and S* from both approaches


---
## Wrap-Up: Method Comparison

| Method | Needs | Convergence | Best for |
|---|---|---|---|
| Gradient Descent | gradient only | slow, robust | huge/noisy problems (deep learning) |
| Newton's Method | gradient + Hessian | very fast (quadratic) | small, smooth problems |
| BFGS (quasi-Newton) | gradient only, builds curvature estimate | fast | general-purpose default |
| Projected Gradient Descent | gradient + easy-to-project set | as fast as GD | box/simplex/ball constraints |
| Penalty Method | gradient of penalized objective | depends on $\mu$ schedule | when projection is hard |
| Lagrange / KKT (scipy `SLSQP`) | gradient + constraint gradients | fast | equality/inequality constraints, general sets |

### Final 🤔 Think About It
Every method you built today assumed you could compute (or approximate) a gradient. What
would you do if `f` were a **black box** — e.g., the output of a physical experiment or a
simulator you can only *query*, with no formula and no gradient available? (This question
has a name: **derivative-free / black-box optimization**. If you're curious, look up
*Nelder-Mead* and *Bayesian Optimization*.)

**You've just built, by hand, the same core ideas that train every neural network, price
every derivative, and route every delivery truck on Earth. Welcome to optimization.**
